# XAI — sparse-transcoder attribution on the KB+QA arm

**Purpose.** Reproduce, on the Kineret cohort, the token-level
attribution analysis reported for INTERVenE-Enc on MIMIC-IV (AAAI
companion paper). Fit sparse transcoders on the trained KB+QA
encoder's per-block MLPs, attach the gradient-decoupled hooks, and
aggregate per-token contributions per outcome across ~300 test
patients.

**Vendored module.** The XAI pipeline was ported from the AAAI
code snapshot at
`../../papers-drafts/AAAI2027/AAAI_2027___INTERVenE/code/INTERVenE/encoder/xai/`
into `kineret/intervene/xai/` (with the `xai.*` imports rewritten to
`kineret.intervene.xai.*`). The demo notebook from the AAAI snapshot
is preserved alongside as `xai_aaai_reference.ipynb`; it is the
authoritative walkthrough of the same API on MIMIC-IV.

**Structure.**

1. Load the trained KB+QA checkpoint + tokenizer.
2. Build the test loader for the KB+QA arm.
3. Cache each block's MLP inputs and outputs (`capture_mlp_io`).
4. Fit one JumpReLU transcoder per encoder block.
5. Attach the gradient-decoupled hooks and compute per-token
   contributions per outcome.
6. Aggregate into top-drivers tables and plot.
7. Export.

**Runtime.** Transcoder training is the expensive step -- a few
epochs per layer on the cached activations. Budget an evening on the
A5000. Attribution itself is a single forward+backward pass per
patient batch, so the aggregation over 300 test patients is quick.

In [ ]:
import os, gc, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=UserWarning)

from kineret.config import paths
from kineret.config import data_config as C
from kineret.cohort import Cohort
from kineret.benchmark import ensure_prepared, resolve_device, run_dir_for

# Vendored XAI stack.
from kineret.intervene.xai import (
    TranscoderHookManager,
    feature_activations,
    feature_to_logit_attribution,
    collect_feature_activations,
    feature_concept_table,
)
from kineret.intervene.xai.transcoder import (
    capture_mlp_io,
    train_transcoders,
    save_transcoders,
    load_transcoders,
)

# Kineret's INTERVenE build helpers.
from kineret.intervene.config import dataset_config as DCFG
from kineret.intervene.config import model_config as MCFG

OUT_ROOT   = paths.OUTPUT_ROOT
CKPT_ROOT  = paths.CHECKPOINT_DIR
FIG_DIR    = os.path.join(OUT_ROOT, 'figures', 'xai')
XAI_CACHE  = os.path.join(paths.PROCESSED_DIR, 'xai_kb_qa')  # transcoders + acts
os.makedirs(FIG_DIR,   exist_ok=True)
os.makedirs(XAI_CACHE, exist_ok=True)

ARM_KEY    = 'intervene_kb_qa'
USE_QA     = True
K          = C.EVAL_CONTEXT_DAYS
N_PATIENTS = 300           # matches the MIMIC-IV analysis
DEVICE     = resolve_device(None, verbose=True)

## 1 — Load checkpoint + build the KB+QA test loader

Re-uses the arm's own configuration path so the tokenizer, vocabulary,
and processed splits are identical to what the trained model saw.

In [ ]:
cohort = ensure_prepared()

# Point the ported INTERVenE package at this (K, QA) cell -- same call as
# kineret.intervene.train.run() makes.
DCFG.configure(K, USE_QA, outcome_names=cohort.outcome_names)
MCFG.reset_to_defaults()
MCFG.configure_windows(K)
MCFG.configure_checkpoints(K, USE_QA, root=CKPT_ROOT)

# Late-bind (the ported package uses `from ... import *` at import time).
from kineret.intervene.dataset     import EMRTokenizer, get_dataloader, EMRDataset, collate_emr
from kineret.intervene.transformer import InterveneEncoder
from kineret.intervene.embedder    import EMREmbedding

ckpt_path = os.path.join(MCFG.MODEL_CONFIG['ckpt_dir'],
                          MCFG.MODEL_CONFIG['phase3_ckpt_best'])
print(f'Loading checkpoint: {ckpt_path}')

# Load model. InterveneEncoder.load returns (model, tokenizer_state, ...);
# see kineret/intervene/transformer.py::InterveneEncoder.load for the exact
# signature. On any mismatch check the AAAI demo (xai_aaai_reference.ipynb)
# and mirror its loading pattern.
state = torch.load(ckpt_path, map_location=DEVICE)
# The exact constructor + load call depends on how the arm saves; the
# helper below wraps the standard load path.
tokenizer, model = None, None   # populated below
raise NotImplementedError(
    'Fill in the checkpoint-load call to match kineret.intervene.train.run().'
    ' The trained model is at `ckpt_path`; instantiate InterveneEncoder with'
    ' MCFG.MODEL_CONFIG and load `state`. See xai_aaai_reference.ipynb for the'
    ' MIMIC-IV equivalent.'
)
model.eval().to(DEVICE)

In [ ]:
# Build the same test dataloader kineret.intervene.train.run() uses at
# scoring time. Reuse `_prepare_split` from kineret.intervene.train to
# avoid duplicating the alias/scaler wiring.
from kineret.intervene.train import _prepare_split, _build_label_frames

abstract_df = pd.read_parquet(paths.PROCESSED_DIR + '/mediator_output_norm.parquet') \
    if os.path.exists(paths.PROCESSED_DIR + '/mediator_output_norm.parquet') \
    else None
# NOTE: `_prepare_split` is the exact function the arm calls at inference.
# It returns (dataloader, ...); adapt the call here to the current signature
# by consulting kineret/intervene/train.py::run.
raise NotImplementedError(
    'Build the KB+QA test dataloader here using kineret.intervene.train._prepare_split.'
    ' Trim to N_PATIENTS = 300 for the attribution aggregation (matches the AAAI'
    ' analysis size).'
)
# test_loader = ...

## 2 — Cache MLP inputs/outputs per block

`capture_mlp_io` attaches forward hooks and streams pre/post-MLP
activations to disk (under `XAI_CACHE/layer_<L>.pt`). Do this once,
then reuse for training.

In [ ]:
n_layers = len(model.blocks)
layer_indices = list(range(n_layers))

print(f'Capturing MLP I/O over {N_PATIENTS} patients, {n_layers} layers...')
capture_mlp_io(
    model=model,
    dataloader=test_loader,
    layer_indices=layer_indices,
    cache_dir=XAI_CACHE,
    device=DEVICE,
    max_patients=N_PATIENTS,
)

## 3 — Fit one JumpReLU transcoder per block

Sparse code + reconstruction loss + L0 penalty; the AAAI paper's
target is per-layer $R^2 \gtrsim 0.97$ with a sparse per-token code.

In [ ]:
transcoders = train_transcoders(
    layer_indices=layer_indices,
    cache_dir=XAI_CACHE,
    device=DEVICE,
    d_feat=None,        # default: 8 * d_mlp (see xai/transcoder/train.py)
    n_epochs=None,      # default per AAAI config
)
save_transcoders(transcoders, XAI_CACHE)
print(f'{len(transcoders)} transcoders trained + saved to {XAI_CACHE}')

## 4 — Gradient-decoupled attribution per outcome

The identity
$y^{\mathrm{used}}_L = y^{\mathrm{true}}_L + (\hat{y}_L - \mathrm{sg}(\hat{y}_L))$
keeps the forward pass bit-exact to the deployed model while making
$\partial \mathrm{risk}_k / \partial f_L$ well-defined for gradient-based
attribution.

In [ ]:
with TranscoderHookManager(model, transcoders) as hooks:
    contribs = feature_to_logit_attribution(
        model=model,
        dataloader=test_loader,
        outcomes=cohort.outcome_names,
        device=DEVICE,
        max_patients=N_PATIENTS,
    )

# `contribs` is a dict outcome -> DataFrame(token, contribution) aggregated
# per token across the sampled patient set.
for outcome, df in contribs.items():
    df.to_csv(os.path.join(FIG_DIR, f'contrib_{outcome}.csv'), index=False)
print(f'Wrote {len(contribs)} contribution tables to {FIG_DIR}')

## 5 — Top drivers per outcome — the paper figure

One panel per outcome; top-$k$ positive and negative token contributions.

In [ ]:
TOP_K = 12
outcomes = list(contribs.keys())
cols = 2
rows = (len(outcomes) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(9, 3.2 * rows), squeeze=False)
for ax, outcome in zip(axes.flat, outcomes):
    df = contribs[outcome].copy()
    top_pos = df.nlargest(TOP_K, 'contribution')
    top_neg = df.nsmallest(TOP_K, 'contribution')
    sub = pd.concat([top_neg, top_pos]).sort_values('contribution')
    colors = ['#d62728' if v > 0 else '#1f77b4' for v in sub['contribution']]
    ax.barh(sub['token'], sub['contribution'], color=colors)
    ax.axvline(0, color='k', lw=0.4)
    ax.set_title(outcome, fontsize=9)
    ax.tick_params(axis='y', labelsize=7)
for ax in axes.flat[len(outcomes):]:
    ax.axis('off')
fig.suptitle('Top per-token risk drivers (KB+QA arm) — red pushes risk up',
              y=1.01, fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'xai_top_drivers.png'), dpi=200,
             bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'xai_top_drivers.pdf'),
             bbox_inches='tight')
fig

## Notes

- **Two cells raise `NotImplementedError` deliberately** — the
  checkpoint-loading and dataloader-construction glue is arm-specific
  and mirrors the same lines in `kineret.intervene.train.run()`.
  Copy those two sections from `run()` (the checkpoint load, and the
  `_prepare_split` call for the test split) into the marked cells
  before running the notebook.
- **AAAI reference:** `xai_aaai_reference.ipynb` is the demo that ships
  with the vendored code, run on the MIMIC-IV KB+QA arm. It uses the
  identical API and is the authoritative example of the sequence
  (capture → train → attach → attribute → plot).
- **`*_PATTERN` tokens.** The AIIM Discussion claims compliance-pattern
  tokens rank among the top drivers for seven of eight outcomes. Add
  a filter step here to isolate those tokens and count how many
  outcomes place at least one `_PATTERN` token in the top $K$.
- **Memory.** Cached MLP I/O for 300 patients across all encoder blocks
  is a few GB; leave headroom on the A5000. Reduce `N_PATIENTS` if
  short.